# Scientific Image Forgery Localization — Comparable Training Pipeline

Bu notebook, bilimsel görüntülerde **pixel-level forgery localization** problemi için uçtan uca bir PyTorch segmentation pipeline'ı kurar.

## Varsayılan ilk deney
- **Model:** `DeepLabV3Plus`
- **Encoder:** `tu-efficientnet_b5`
- **Loss:** `BCEWithLogits + Dice`
- **Split:** `case_id` bazlı `GroupShuffleSplit`
- **Artifact tracking:** her çalıştırmada `runs/{run_name}/...`

## Kaydedilen çıktılar
- `config.json`
- `split_summary.json`
- `train_history.csv`
- `best_metrics.json`
- `threshold_sweep.csv`
- `best_model.pth`
- `last_model.pth`
- `oof_val_predictions.npz`
- `submission.csv`
- eğitim grafikleri ve validation görselleştirmeleri

## Notebook akışı
1. Gerekli Kütüphaneleri İçe Aktar
2. Örnek Girdi ve Beklenen Çıktıyı Tanımla
3. Temel Mantığı Fonksiyonlara Dönüştür
4. Hata Kontrolleri ve Kenar Durumları Ekle
5. Fonksiyonları Test Verileriyle Çalıştır
6. Sonucu Modüler ve Yeniden Kullanılabilir Hale Getir

In [ ]:
import os
import re
import gc
import sys
import json
import time
import random
import warnings
import subprocess
import contextlib
import importlib.util
from pathlib import Path
from datetime import datetime
from collections import defaultdict


def ensure_package(pip_name, import_name=None):
    import_name = import_name or pip_name
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


for pip_name, import_name in [
    ("segmentation-models-pytorch", "segmentation_models_pytorch"),
    ("albumentations", "albumentations"),
    ("timm", "timm"),
    ("opencv-python-headless", "cv2"),
]:
    ensure_package(pip_name, import_name)

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

from sklearn.model_selection import GroupShuffleSplit

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Örnek Girdi ve Beklenen Çıktıyı Tanımla

Beklenen veri yapısı:

```text
train_images/
  authentic/
  forged/
train_masks/
supplemental_images/
supplemental_masks/
test_images/
```

Beklenen submission formatı:
- `case_id`
- `annotation`

Boş tahminler için `annotation = "authentic"`; dolu tahminler için RLE string üretilecektir.

In [ ]:
IMG_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

CFG = {
    "seed": 42,
    "data_root": "/kaggle/input/datasets/koushikkumardinda/scientific-image-forgery-detection/recodai-luc-scientific-image-forgery-detection",
    "model_name": "deeplabv3plus",          # alternatives: unetplusplus, unet
    "encoder_name": "tu-efficientnet_b5",   # easy to swap later
    "encoder_weights": "imagenet",
    "loss_name": "bce_dice",                # alternative: focal_dice
    "img_size": 512 if torch.cuda.is_available() else 320,
    "batch_size": 4 if torch.cuda.is_available() else 2,
    "epochs": 20,
    "lr": 2e-4,
    "weight_decay": 1e-4,
    "num_workers": 2 if (os.name != "nt" and torch.cuda.is_available()) else 0,
    "val_size": 0.20,
    "amp": True,
    "bce_weight": 0.4,
    "dice_weight": 0.6,
    "focal_weight": 0.3,
    "positive_sample_weight": 3.0,
    "use_weighted_sampler": True,
    "thresholds": [round(float(x), 2) for x in np.arange(0.10, 0.91, 0.05)],
    "monitor_metric": "dice",
    "early_stopping_patience": 6,
    "grad_clip": 1.0,
    "min_delta": 1e-4,
    "positive_fraction_samples": 96,
    "max_pos_weight": 50.0,
    "tta_inference": True,
    "min_component_area": 12,
    "rle_order": "C",   # change if competition expects a different flatten order
    "runs_root": None,
    "run_name": None,
}


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True


def resolve_data_root(custom_root=None):
    candidates = []
    if custom_root:
        candidates.append(Path(custom_root))

    kaggle_inputs = [Path("/kaggle/input"), Path("/kaggle/working")]
    for root in kaggle_inputs:
        if root.exists():
            candidates.extend([p for p in root.rglob("*") if p.is_dir()])

    colab_candidates = [Path("/content"), Path("/content/drive/MyDrive")]
    for root in colab_candidates:
        if root.exists():
            candidates.extend([root] + [p for p in root.rglob("*") if p.is_dir()])

    cwd = Path.cwd()
    candidates.extend([cwd, cwd.parent])

    seen = set()
    for cand in candidates:
        cand = Path(cand)
        if cand in seen:
            continue
        seen.add(cand)
        if (cand / "train_images").exists() and (cand / "test_images").exists():
            return cand
    return None


def ensure_run_directories(cfg):
    if cfg.get("runs_root"):
        runs_root = Path(cfg["runs_root"]).expanduser()
    elif Path("/kaggle/working").exists():
        runs_root = Path("/kaggle/working/runs")
    elif Path("/content").exists():
        runs_root = Path("/content/runs")
    else:
        runs_root = Path.cwd() / "runs"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_encoder = cfg["encoder_name"].replace("/", "-")
    run_name = cfg["run_name"] or f"{timestamp}_{cfg['model_name']}_{safe_encoder}_{cfg['loss_name']}"

    run_dir = runs_root / run_name
    plots_dir = run_dir / "plots"
    visualizations_dir = run_dir / "visualizations"
    for path in [runs_root, run_dir, plots_dir, visualizations_dir]:
        path.mkdir(parents=True, exist_ok=True)

    return run_name, {
        "runs_root": runs_root,
        "run_dir": run_dir,
        "plots_dir": plots_dir,
        "visualizations_dir": visualizations_dir,
        "registry_path": runs_root / "experiment_registry.csv",
        "best_model_path": run_dir / "best_model.pth",
        "last_model_path": run_dir / "last_model.pth",
        "history_path": run_dir / "train_history.csv",
        "metrics_path": run_dir / "best_metrics.json",
        "split_summary_path": run_dir / "split_summary.json",
        "threshold_path": run_dir / "threshold_sweep.csv",
        "oof_path": run_dir / "oof_val_predictions.npz",
        "submission_path": run_dir / "submission.csv",
    }


DATA_ROOT = resolve_data_root(CFG.get("data_root"))
assert DATA_ROOT is not None, "Dataset root not found. Set CFG['data_root'] manually for Kaggle/Colab/local." 

RUN_NAME, PATHS = ensure_run_directories(CFG)
CFG["data_root"] = str(DATA_ROOT)
CFG["run_name"] = RUN_NAME

with open(PATHS["run_dir"] / "config.json", "w", encoding="utf-8") as f:
    json.dump(CFG, f, indent=2)

print("DEVICE:", DEVICE)
print("DATA_ROOT:", DATA_ROOT)
print("RUN_DIR:", PATHS["run_dir"])

display(pd.DataFrame([
    {"example_case_id": 1001, "expected_annotation": "authentic"},
    {"example_case_id": 1002, "expected_annotation": "12 5 40 3 ..."},
]))

## 3. Temel Mantığı Fonksiyonlara Dönüştür

Bu bölümde veri indeksleme, maske birleştirme, metrik hesaplama ve kayıt yardımcıları tanımlanır.

## 4. Hata Kontrolleri ve Kenar Durumları Ekle

Fonksiyonlar; eksik dosya, boş veri, okunamayan görsel ve `case_id` çıkarılamayan dosya adları gibi durumlar için korumalı olacak şekilde yazılmıştır.

In [ ]:
def save_json(path, obj):
    def default(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        return str(o)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=default)


def extract_case_id_from_name(path_obj):
    stem = Path(path_obj).stem
    match = re.match(r"^(\d+)", stem)
    if not match:
        raise ValueError(f"Could not extract case_id from filename: {path_obj}")
    return int(match.group(1))


def list_image_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMG_SUFFIXES])


def read_image_rgb(path):
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")

    if img.dtype == np.uint16:
        img = cv2.convertScaleAbs(img, alpha=255.0 / 65535.0)

    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
    else:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img


def load_mask_array(mask_path):
    arr = np.asarray(np.load(mask_path))
    if arr.ndim == 3:
        arr = arr[..., 0]
    arr = np.nan_to_num(arr, nan=0.0)
    return (arr > 0).astype(np.uint8)


def union_masks(mask_paths, image_shape_hw):
    h, w = image_shape_hw
    merged = np.zeros((h, w), dtype=np.uint8)
    for mask_path in mask_paths:
        mask = load_mask_array(mask_path)
        if mask.shape[:2] != (h, w):
            mask = cv2.resize(mask.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
        merged = np.maximum(merged, mask)
    return merged


def rle_encode(mask, order="C"):
    mask = (mask > 0).astype(np.uint8)
    pixels = mask.flatten(order=order)
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return "authentic" if len(runs) == 0 else " ".join(map(str, runs))


def postprocess_mask(mask, min_area=0):
    mask = (mask > 0).astype(np.uint8)
    if min_area <= 0 or mask.sum() == 0:
        return mask

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    cleaned = np.zeros_like(mask)
    for lab in range(1, num_labels):
        area = stats[lab, cv2.CC_STAT_AREA]
        if area >= min_area:
            cleaned[labels == lab] = 1
    return cleaned


def pixel_metrics_from_arrays(y_true, y_pred, eps=1e-7):
    yt = y_true.reshape(-1).astype(np.uint8)
    yp = y_pred.reshape(-1).astype(np.uint8)

    tp = np.logical_and(yt == 1, yp == 1).sum(dtype=np.float64)
    fp = np.logical_and(yt == 0, yp == 1).sum(dtype=np.float64)
    fn = np.logical_and(yt == 1, yp == 0).sum(dtype=np.float64)

    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = (2 * precision * recall) / (precision + recall + eps)
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "dice": float(dice),
        "iou": float(iou),
        "pred_pos_rate": float(yp.mean()),
        "true_pos_rate": float(yt.mean()),
    }


def soft_dice_from_probs_np(y_true, y_prob, eps=1e-7):
    y_true = y_true.reshape(-1).astype(np.float32)
    y_prob = y_prob.reshape(-1).astype(np.float32)
    inter = (y_true * y_prob).sum()
    return float((2 * inter + eps) / (y_true.sum() + y_prob.sum() + eps))


def evaluate_thresholds(all_masks, all_probs, thresholds):
    masks = all_masks[:, 0] if all_masks.ndim == 4 else all_masks
    probs = all_probs[:, 0] if all_probs.ndim == 4 else all_probs

    rows = []
    for thr in thresholds:
        preds = (probs >= thr).astype(np.uint8)
        metrics = pixel_metrics_from_arrays(masks, preds)
        metrics["threshold"] = float(thr)
        rows.append(metrics)

    sweep_df = pd.DataFrame(rows).sort_values("threshold").reset_index(drop=True)
    best_idx = sweep_df["dice"].idxmax()
    best_threshold = float(sweep_df.loc[best_idx, "threshold"])
    best_metrics = sweep_df.loc[best_idx].to_dict()
    best_metrics["soft_dice"] = float(soft_dice_from_probs_np(masks, probs))
    return sweep_df, best_threshold, best_metrics


def index_mask_files(mask_dir):
    mask_dir = Path(mask_dir)
    mask_map = defaultdict(list)
    if not mask_dir.exists():
        return mask_map
    for p in sorted(mask_dir.rglob("*.npy")):
        cid = extract_case_id_from_name(p.name)
        mask_map[cid].append(str(p))
    return mask_map


def build_samples_dataframe(data_root):
    data_root = Path(data_root)
    train_mask_map = index_mask_files(data_root / "train_masks")
    supp_mask_map = index_mask_files(data_root / "supplemental_masks")

    rows = []

    for p in list_image_files(data_root / "train_images" / "authentic"):
        cid = extract_case_id_from_name(p.name)
        rows.append({
            "case_id": cid,
            "image_path": str(p),
            "source": "train",
            "label": "authentic",
            "mask_paths": [],
            "has_mask": 0,
            "is_forged": 0,
        })

    for p in list_image_files(data_root / "train_images" / "forged"):
        cid = extract_case_id_from_name(p.name)
        mask_paths = train_mask_map.get(cid, [])
        rows.append({
            "case_id": cid,
            "image_path": str(p),
            "source": "train",
            "label": "forged",
            "mask_paths": list(mask_paths),
            "has_mask": int(len(mask_paths) > 0),
            "is_forged": 1,
        })

    for p in list_image_files(data_root / "supplemental_images"):
        cid = extract_case_id_from_name(p.name)
        mask_paths = supp_mask_map.get(cid, [])
        rows.append({
            "case_id": cid,
            "image_path": str(p),
            "source": "supplemental",
            "label": "forged" if len(mask_paths) > 0 else "authentic",
            "mask_paths": list(mask_paths),
            "has_mask": int(len(mask_paths) > 0),
            "is_forged": int(len(mask_paths) > 0),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No training samples found under: {data_root}")

    df = df.drop_duplicates(subset=["image_path"]).sort_values(["case_id", "image_path"]).reset_index(drop=True)
    missing_masks = df[(df["label"] == "forged") & (df["has_mask"] == 0)]
    if len(missing_masks) > 0:
        print(f"Warning: {len(missing_masks)} forged samples do not have a mask file. They will behave like zero-mask examples.")

    return df


def build_test_dataframe(data_root):
    data_root = Path(data_root)
    rows = []
    for p in list_image_files(data_root / "test_images"):
        rows.append({
            "case_id": extract_case_id_from_name(p.name),
            "image_path": str(p),
        })
    test_df = pd.DataFrame(rows).sort_values("case_id").reset_index(drop=True)
    if test_df.empty:
        print("No test images found; submission step will be skipped.")
    return test_df


def estimate_positive_fraction(df, max_samples=96):
    if df.empty:
        return 0.0
    sample_df = df.sample(n=min(len(df), max_samples), random_state=CFG["seed"])
    positive_pixels = 0.0
    total_pixels = 0.0
    for row in sample_df.itertuples(index=False):
        image = read_image_rgb(row.image_path)
        if row.has_mask:
            mask = union_masks(row.mask_paths, image.shape[:2])
        else:
            mask = np.zeros(image.shape[:2], dtype=np.uint8)
        positive_pixels += float(mask.sum())
        total_pixels += float(mask.size)
    return positive_pixels / max(total_pixels, 1.0)


def make_overlay(image, mask, alpha=0.35):
    image = image.copy()
    mask = (mask > 0).astype(np.uint8)
    color = np.zeros_like(image)
    color[..., 0] = 255
    return np.where(mask[..., None] > 0, (1 - alpha) * image + alpha * color, image).astype(np.uint8)


def save_history_plots(history_df, plots_dir):
    if history_df.empty:
        return

    fig, ax = plt.subplots(1, 1, figsize=(7, 4))
    ax.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    ax.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    ax.set_title("Loss Curve")
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(Path(plots_dir) / "loss_curve.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(1, 1, figsize=(7, 4))
    for col in ["val_dice", "val_iou", "val_f1"]:
        if col in history_df.columns:
            ax.plot(history_df["epoch"], history_df[col], label=col)
    ax.set_title("Validation Metrics")
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(Path(plots_dir) / "metric_curve.png", dpi=150)
    plt.close(fig)


def save_threshold_plot(sweep_df, plots_dir):
    if sweep_df.empty:
        return
    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    for col in ["dice", "iou", "f1", "precision", "recall"]:
        if col in sweep_df.columns:
            ax.plot(sweep_df["threshold"], sweep_df[col], marker="o", label=col)
    ax.set_title("Threshold Sweep")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Metric")
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2)
    fig.tight_layout()
    fig.savefig(Path(plots_dir) / "threshold_curve.png", dpi=150)
    plt.close(fig)

## 5. Fonksiyonları Test Verileriyle Çalıştır

Önce dataframe oluşturulacak, ardından leakage-safe split doğrulanacak ve birkaç örnek görsel kaydedilecektir.

In [ ]:
def get_preprocessing_fn(cfg):
    return smp.encoders.get_preprocessing_fn(cfg["encoder_name"], cfg["encoder_weights"])


def get_transforms(cfg, is_train=True, preprocessing_fn=None):
    transforms = [
        A.LongestMaxSize(max_size=int(cfg["img_size"])),
        A.PadIfNeeded(
            min_height=int(cfg["img_size"]),
            min_width=int(cfg["img_size"]),
            border_mode=cv2.BORDER_CONSTANT,
            value=0,
            mask_value=0,
        ),
    ]

    if is_train:
        transforms.extend([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.05,
                scale_limit=0.10,
                rotate_limit=20,
                border_mode=cv2.BORDER_CONSTANT,
                value=0,
                mask_value=0,
                p=0.50,
            ),
            A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.35),
            A.GaussNoise(var_limit=(10.0, 40.0), p=0.15),
            A.OneOf([
                A.GaussianBlur(blur_limit=(3, 5), p=1.0),
                A.MotionBlur(blur_limit=3, p=1.0),
            ], p=0.15),
        ])

    if preprocessing_fn is not None:
        transforms.append(A.Lambda(image=lambda x, **kwargs: preprocessing_fn(x)))

    transforms.append(ToTensorV2(transpose_mask=True))
    return A.Compose(transforms)


class ForgeryDataset(Dataset):
    def __init__(self, df, cfg, is_train=True, with_masks=True):
        self.df = df.reset_index(drop=True).copy()
        self.cfg = cfg
        self.is_train = is_train
        self.with_masks = with_masks
        self.preprocessing_fn = get_preprocessing_fn(cfg)
        self.transform = get_transforms(cfg, is_train=is_train, preprocessing_fn=self.preprocessing_fn)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = read_image_rgb(row["image_path"])
        orig_h, orig_w = image.shape[:2]

        if self.with_masks and row.get("has_mask", 0):
            mask = union_masks(row["mask_paths"], image.shape[:2])
        else:
            mask = np.zeros(image.shape[:2], dtype=np.uint8)

        transformed = self.transform(image=image, mask=mask)
        image_tensor = transformed["image"].float()
        mask_tensor = transformed["mask"].float()
        if mask_tensor.ndim == 2:
            mask_tensor = mask_tensor.unsqueeze(0)
        mask_tensor = (mask_tensor > 0).float()

        return {
            "image": image_tensor,
            "mask": mask_tensor,
            "case_id": torch.tensor(int(row["case_id"]), dtype=torch.long),
            "image_path": row["image_path"],
            "orig_hw": torch.tensor([orig_h, orig_w], dtype=torch.long),
        }


def make_group_split(df, val_size=0.2, seed=42):
    splitter = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
    train_idx, val_idx = next(splitter.split(df, groups=df["case_id"]))
    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)
    return train_df, val_df


def validate_group_split(train_df, val_df):
    overlap = set(train_df["case_id"].tolist()).intersection(set(val_df["case_id"].tolist()))
    assert len(overlap) == 0, f"Leakage detected. Overlapping case_ids: {sorted(list(overlap))[:10]}"
    print("Leakage check passed. Overlapping case_ids:", len(overlap))


def make_dataloaders(train_df, val_df, cfg):
    train_ds = ForgeryDataset(train_df, cfg, is_train=True, with_masks=True)
    val_ds = ForgeryDataset(val_df, cfg, is_train=False, with_masks=True)

    sampler = None
    shuffle = True
    if cfg.get("use_weighted_sampler", True):
        sample_weights = np.where(train_df["is_forged"].values == 1, cfg["positive_sample_weight"], 1.0).astype(np.float64)
        sampler = WeightedRandomSampler(torch.as_tensor(sample_weights, dtype=torch.double), len(sample_weights), replacement=True)
        shuffle = False

    train_loader = DataLoader(
        train_ds,
        batch_size=int(cfg["batch_size"]),
        sampler=sampler,
        shuffle=shuffle if sampler is None else False,
        num_workers=int(cfg["num_workers"]),
        pin_memory=DEVICE.type == "cuda",
        drop_last=False,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=int(cfg["batch_size"]),
        shuffle=False,
        num_workers=int(cfg["num_workers"]),
        pin_memory=DEVICE.type == "cuda",
        drop_last=False,
    )
    return train_loader, val_loader, train_ds, val_ds


def show_dataset_samples(df, save_path, n=4):
    if df.empty:
        return
    rng = np.random.default_rng(CFG["seed"])
    pick_n = min(n, len(df))
    indices = rng.choice(len(df), size=pick_n, replace=False)

    fig, axes = plt.subplots(pick_n, 3, figsize=(12, 4 * pick_n))
    if pick_n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_ax, idx in zip(axes, indices):
        row = df.iloc[int(idx)]
        image = read_image_rgb(row["image_path"])
        mask = union_masks(row["mask_paths"], image.shape[:2]) if row["has_mask"] else np.zeros(image.shape[:2], dtype=np.uint8)
        overlay = make_overlay(image, mask)

        row_ax[0].imshow(image)
        row_ax[0].set_title(f"case_id={row['case_id']} | {row['label']}")
        row_ax[1].imshow(mask, cmap="gray")
        row_ax[1].set_title("mask")
        row_ax[2].imshow(overlay)
        row_ax[2].set_title("overlay")
        for ax in row_ax:
            ax.axis("off")

    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.show()
    plt.close(fig)


samples_df = build_samples_dataframe(DATA_ROOT)
train_df, val_df = make_group_split(samples_df, val_size=CFG["val_size"], seed=CFG["seed"])
validate_group_split(train_df, val_df)

split_summary = {
    "n_total": int(len(samples_df)),
    "n_train": int(len(train_df)),
    "n_val": int(len(val_df)),
    "train_unique_cases": int(train_df["case_id"].nunique()),
    "val_unique_cases": int(val_df["case_id"].nunique()),
    "train_positive_images": int(train_df["is_forged"].sum()),
    "val_positive_images": int(val_df["is_forged"].sum()),
}
save_json(PATHS["split_summary_path"], split_summary)

display(pd.DataFrame([split_summary]))
display(samples_df.head())

CFG["estimated_positive_fraction"] = float(estimate_positive_fraction(train_df, max_samples=CFG["positive_fraction_samples"]))
CFG["pos_weight"] = float(min(CFG["max_pos_weight"], max(1.0, (1.0 - max(CFG["estimated_positive_fraction"], 1e-6)) / max(CFG["estimated_positive_fraction"], 1e-6))))
save_json(PATHS["run_dir"] / "config.json", CFG)

train_loader, val_loader, train_ds, val_ds = make_dataloaders(train_df, val_df, CFG)
show_dataset_samples(train_df, PATHS["visualizations_dir"] / "dataset_preview.png", n=4)

print(f"Estimated positive pixel fraction: {CFG['estimated_positive_fraction']:.8f}")
print(f"Using pos_weight: {CFG['pos_weight']:.3f}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# Hücre 1 — forged örneklerde ground truth / maske kontrolü

def inspect_forged_ground_truth_status(df, split_name="dataset", max_rows=20):
    forged_df = df[df["label"] == "forged"].copy().reset_index(drop=True)
    if forged_df.empty:
        print(f"{split_name} içinde forged örnek bulunamadı.")
        return pd.DataFrame(), pd.DataFrame()

    gt_positive_pixels = []
    gt_is_empty = []
    n_mask_files = []

    for row in forged_df.itertuples(index=False):
        n_mask_files.append(len(row.mask_paths))
        if row.has_mask:
            image = read_image_rgb(row.image_path)
            gt_mask = union_masks(row.mask_paths, image.shape[:2])
            pos_pixels = int(gt_mask.sum())
        else:
            pos_pixels = 0
        gt_positive_pixels.append(pos_pixels)
        gt_is_empty.append(int(pos_pixels == 0))

    forged_df["n_mask_files"] = n_mask_files
    forged_df["gt_positive_pixels"] = gt_positive_pixels
    forged_df["gt_is_empty"] = gt_is_empty

    summary_df = pd.DataFrame([
        {
            "split": split_name,
            "forged_total": int(len(forged_df)),
            "forged_with_mask_file": int((forged_df["has_mask"] == 1).sum()),
            "forged_without_mask_file": int((forged_df["has_mask"] == 0).sum()),
            "forged_with_nonempty_gt": int((forged_df["gt_positive_pixels"] > 0).sum()),
            "forged_with_empty_gt": int((forged_df["gt_positive_pixels"] == 0).sum()),
        }
    ])

    print(f"\n[{split_name}] forged ground truth kontrol özeti")
    display(summary_df)

    suspicious_df = forged_df[
        (forged_df["has_mask"] == 0) | (forged_df["gt_positive_pixels"] == 0)
    ][[
        "case_id", "source", "label", "has_mask", "n_mask_files", "gt_positive_pixels", "image_path"
    ]].sort_values(["has_mask", "gt_positive_pixels", "case_id"]).reset_index(drop=True)

    if suspicious_df.empty:
        print("Şüpheli kayıt yok: forged örneklerin ground truth maskeleri dolu görünüyor.")
    else:
        print("Ground truth'u siyah çıkabilecek forged örnekler:")
        display(suspicious_df.head(max_rows))

    return summary_df, forged_df


train_forged_summary_df, train_forged_audit_df = inspect_forged_ground_truth_status(
    samples_df,
    split_name="all_samples",
    max_rows=20,
)

val_forged_summary_df, val_forged_audit_df = inspect_forged_ground_truth_status(
    val_df,
    split_name="validation",
    max_rows=20,
)

# Hücre 2 — maskesi bulunan forged validation görüntülerde en iyi ağırlıklarla tahmin

@torch.no_grad()
def predict_on_masked_forged_validation_with_best_ckpt(
    val_df,
    cfg,
    device,
    checkpoint_path=None,
    num_samples=6,
    threshold=None,
    compare_threshold=0.50,
    only_nonempty_gt=True,
    use_tta=False,
):
    candidate_df = val_df[(val_df["label"] == "forged") & (val_df["has_mask"] == 1)].copy().reset_index(drop=True)
    if candidate_df.empty:
        print("Validation split içinde maskesi bulunan forged örnek yok.")
        return None

    if only_nonempty_gt:
        positive_keep = []
        for _, row in candidate_df.iterrows():
            image = read_image_rgb(row["image_path"])
            gt_mask = union_masks(row["mask_paths"], image.shape[:2])
            positive_keep.append(int(gt_mask.sum()) > 0)
        candidate_df = candidate_df[np.array(positive_keep, dtype=bool)].reset_index(drop=True)

    if candidate_df.empty:
        print("Mask dosyası olan ama ground truth'u boş olmayan forged validation örneği bulunamadı.")
        return None

    ckpt_candidates = []
    if checkpoint_path is not None:
        ckpt_candidates.append(Path(checkpoint_path))
    if "PATHS" in globals():
        ckpt_candidates.append(Path(PATHS["best_model_path"]))
    ckpt_candidates.extend([
        Path.cwd() / "deeplabv3+" / "best_model.pth",
        Path.cwd() / "best_model.pth",
    ])

    ckpt_path = next((p for p in ckpt_candidates if p.exists()), None)
    if ckpt_path is None:
        raise FileNotFoundError("best_model.pth bulunamadı.")

    ckpt = torch.load(ckpt_path, map_location=device)
    model_cfg = dict(cfg)
    if isinstance(ckpt, dict) and "cfg" in ckpt and isinstance(ckpt["cfg"], dict):
        model_cfg.update(ckpt["cfg"])

    model = build_model(model_cfg).to(device)
    state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    model.load_state_dict(state_dict)
    model.eval()

    if threshold is None:
        threshold = float(ckpt.get("threshold", 0.5)) if isinstance(ckpt, dict) else 0.5

    save_dir = Path(PATHS["visualizations_dir"]) / "best_ckpt_masked_forged_only"
    save_dir.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(model_cfg.get("seed", 42))
    pick_n = min(int(num_samples), len(candidate_df))
    chosen_indices = rng.choice(len(candidate_df), size=pick_n, replace=False)
    shown_case_ids = []

    print(f"Checkpoint: {ckpt_path}")
    print(f"Using threshold: {threshold:.2f} | Compare threshold: {compare_threshold:.2f}")
    print(f"Eligible masked forged validation samples: {len(candidate_df)} | Showing: {pick_n}")
    print(f"Saved figures -> {save_dir}")

    for vis_idx, row_idx in enumerate(chosen_indices, start=1):
        row = candidate_df.iloc[int(row_idx)]
        shown_case_ids.append(int(row["case_id"]))

        image = read_image_rgb(row["image_path"])
        gt_mask = union_masks(row["mask_paths"], image.shape[:2])

        one_item_ds = ForgeryDataset(pd.DataFrame([row]), model_cfg, is_train=False, with_masks=True)
        sample = one_item_ds[0]
        image_tensor = sample["image"].unsqueeze(0).to(device)

        prob = predict_with_tta(model, image_tensor, use_tta=use_tta)[0, 0].detach().cpu().numpy()
        prob = cv2.resize(prob, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_LINEAR)

        pred_best = postprocess_mask((prob >= threshold).astype(np.uint8), min_area=model_cfg.get("min_component_area", 0))
        pred_cmp = postprocess_mask((prob >= compare_threshold).astype(np.uint8), min_area=model_cfg.get("min_component_area", 0))

        fig, axes = plt.subplots(1, 6, figsize=(22, 4))
        axes[0].imshow(image)
        axes[0].set_title(f"image | case_id={row['case_id']}")
        axes[1].imshow(gt_mask, cmap="gray")
        axes[1].set_title("ground truth")
        axes[2].imshow(prob, cmap="magma")
        axes[2].set_title("probability")
        axes[3].imshow(pred_best, cmap="gray")
        axes[3].set_title(f"pred @ {threshold:.2f}")
        axes[4].imshow(pred_cmp, cmap="gray")
        axes[4].set_title(f"pred @ {compare_threshold:.2f}")
        axes[5].imshow(make_overlay(image, pred_best))
        axes[5].set_title("overlay @ best_thr")

        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        fig.savefig(save_dir / f"masked_forged_val_{vis_idx:03d}_case_{int(row['case_id'])}.png", dpi=150)
        plt.show()
        plt.close(fig)

    return {
        "save_dir": str(save_dir),
        "shown_case_ids": shown_case_ids,
        "threshold": float(threshold),
    }


masked_forged_pred_info = predict_on_masked_forged_validation_with_best_ckpt(
    val_df=val_df,
    cfg=CFG,
    device=DEVICE,
    checkpoint_path=Path.cwd() / "deeplabv3+" / "best_model.pth",
    num_samples=6,
    threshold=None,
    compare_threshold=0.50,
    only_nonempty_gt=True,
    use_tta=CFG.get("tta_inference", False),
)

masked_forged_pred_info

In [ ]:
# Segmentation için filtrelenmiş train/val split:
# authentic negatifleri KORU, forged örneklerden ise sadece non-empty GT maskesi olanları kullan.


def annotate_ground_truth_columns(df):
    df = df.copy().reset_index(drop=True)
    gt_positive_pixels = []
    gt_is_empty = []
    has_nonempty_gt = []
    n_mask_files = []

    for row in df.itertuples(index=False):
        mask_paths = list(row.mask_paths) if isinstance(row.mask_paths, (list, tuple)) else []
        n_mask_files.append(len(mask_paths))

        pos_pixels = 0
        if getattr(row, "has_mask", 0) and len(mask_paths) > 0:
            image = read_image_rgb(row.image_path)
            gt_mask = union_masks(mask_paths, image.shape[:2])
            pos_pixels = int(gt_mask.sum())

        gt_positive_pixels.append(pos_pixels)
        gt_is_empty.append(int(pos_pixels == 0))
        has_nonempty_gt.append(int(pos_pixels > 0))

    df["n_mask_files"] = n_mask_files
    df["gt_positive_pixels"] = gt_positive_pixels
    df["gt_is_empty"] = gt_is_empty
    df["has_nonempty_gt"] = has_nonempty_gt
    return df


if "samples_df" not in globals():
    samples_df = build_samples_dataframe(DATA_ROOT)

samples_df = annotate_ground_truth_columns(samples_df)
min_gt_positive_pixels = int(CFG.get("min_gt_positive_pixels", 1))

# Keep: authentic negatives + forged samples with non-empty masks
seg_samples_df = samples_df[
    (samples_df["label"] == "authentic") |
    (samples_df["gt_positive_pixels"] >= min_gt_positive_pixels)
].copy().reset_index(drop=True)

dropped_empty_forged_df = samples_df[
    (samples_df["label"] == "forged") &
    (samples_df["gt_positive_pixels"] < min_gt_positive_pixels)
].copy().reset_index(drop=True)

if seg_samples_df.empty:
    raise RuntimeError("Filtered segmentation dataset is empty. No non-empty GT examples were found.")

train_df_filtered, val_df_filtered = make_group_split(
    seg_samples_df,
    val_size=CFG["val_size"],
    seed=CFG["seed"],
)
validate_group_split(train_df_filtered, val_df_filtered)

filtered_split_summary = {
    "n_total_all_samples": int(len(samples_df)),
    "n_used_for_segmentation": int(len(seg_samples_df)),
    "n_dropped_empty_gt_forged": int(len(dropped_empty_forged_df)),
    "train_samples": int(len(train_df_filtered)),
    "val_samples": int(len(val_df_filtered)),
    "train_authentic": int((train_df_filtered["label"] == "authentic").sum()),
    "train_nonempty_gt_forged": int((train_df_filtered["gt_positive_pixels"] > 0).sum()),
    "val_authentic": int((val_df_filtered["label"] == "authentic").sum()),
    "val_nonempty_gt_forged": int((val_df_filtered["gt_positive_pixels"] > 0).sum()),
}

display(pd.DataFrame([filtered_split_summary]))
display(seg_samples_df.head())

filtered_cfg = dict(CFG)
filtered_cfg["train_only_on_nonempty_gt"] = True
filtered_cfg["min_gt_positive_pixels"] = min_gt_positive_pixels
filtered_cfg["estimated_positive_fraction"] = float(
    estimate_positive_fraction(train_df_filtered, max_samples=filtered_cfg["positive_fraction_samples"])
)
filtered_cfg["pos_weight"] = float(
    min(
        filtered_cfg["max_pos_weight"],
        max(
            1.0,
            (1.0 - max(filtered_cfg["estimated_positive_fraction"], 1e-6)) /
            max(filtered_cfg["estimated_positive_fraction"], 1e-6),
        ),
    )
)

train_loader_filtered, val_loader_filtered, train_ds_filtered, val_ds_filtered = make_dataloaders(
    train_df_filtered,
    val_df_filtered,
    filtered_cfg,
)

preview_df = train_df_filtered[train_df_filtered["gt_positive_pixels"] > 0].copy().reset_index(drop=True)
if not preview_df.empty:
    show_dataset_samples(preview_df, PATHS["visualizations_dir"] / "filtered_nonempty_gt_preview.png", n=min(4, len(preview_df)))

print("Filtered segmentation training is ready.")
print(f"Dropped empty-GT forged samples: {len(dropped_empty_forged_df)}")
print(f"Estimated positive pixel fraction: {filtered_cfg['estimated_positive_fraction']:.8f}")
print(f"Using pos_weight: {filtered_cfg['pos_weight']:.3f}")
print(f"Filtered train batches: {len(train_loader_filtered)} | Filtered val batches: {len(val_loader_filtered)}")

In [ ]:
# Filtrelenmiş eğitim için train/validation dağılım istatistikleri

if "train_df_filtered" not in globals() or "val_df_filtered" not in globals():
    raise RuntimeError("Önce filtrelenmiş split hücresini çalıştırın; `train_df_filtered` ve `val_df_filtered` henüz oluşturulmadı.")


def summarize_label_counts(df, split_name):
    authentic_count = int((df["label"] == "authentic").sum())
    forged_count = int((df["label"] == "forged").sum())
    total_count = int(len(df))
    return {
        "split": split_name,
        "total_samples": total_count,
        "authentic_count": authentic_count,
        "forged_count": forged_count,
        "authentic_ratio": round(authentic_count / total_count, 4) if total_count > 0 else 0.0,
        "forged_ratio": round(forged_count / total_count, 4) if total_count > 0 else 0.0,
        "unique_case_ids": int(df["case_id"].nunique()) if "case_id" in df.columns else None,
    }


filtered_label_stats_df = pd.DataFrame([
    summarize_label_counts(train_df_filtered, "train_filtered"),
    summarize_label_counts(val_df_filtered, "validation_filtered"),
])

display(filtered_label_stats_df)

print("Train filtered dağılımı:",
      f"authentic={int((train_df_filtered['label'] == 'authentic').sum())}",
      f"forged={int((train_df_filtered['label'] == 'forged').sum())}")
print("Validation filtered dağılımı:",
      f"authentic={int((val_df_filtered['label'] == 'authentic').sum())}",
      f"forged={int((val_df_filtered['label'] == 'forged').sum())}")

In [ ]:
# Filtrelenmiş split ile yeniden eğitim (authentic + non-empty GT forged)

filtered_cfg = dict(filtered_cfg)
filtered_cfg["run_name"] = (
    f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_"
    f"{filtered_cfg['model_name']}_{filtered_cfg['encoder_name'].replace('/', '-')}_"
    f"{filtered_cfg['loss_name']}_nonemptyGT"
)

FILTERED_RUN_NAME, FILTERED_PATHS = ensure_run_directories(filtered_cfg)
filtered_cfg["run_name"] = FILTERED_RUN_NAME
filtered_cfg["data_root"] = str(DATA_ROOT)

save_json(FILTERED_PATHS["run_dir"] / "config.json", filtered_cfg)

filtered_results = run_experiment(
    filtered_cfg,
    train_loader_filtered,
    val_loader_filtered,
    train_ds_filtered,
    val_ds_filtered,
    DATA_ROOT,
    FILTERED_PATHS,
    DEVICE,
)

print("Filtered artifacts saved under:", filtered_results["run_dir"])
if filtered_results["best_metrics"] is not None:
    display(pd.DataFrame([filtered_results["best_metrics"]]))
if not filtered_results["submission_df"].empty:
    display(filtered_results["submission_df"].head())

filtered_results

In [ ]:
# Eğitim sonrası — en iyi model ağırlığıyla validation üzerinde 3 authentic + 3 forged tahmin görselleştirme

if "DEVICE" not in globals():
    raise RuntimeError("Önce kurulum/eğitim hücrelerini çalıştırın; `DEVICE` henüz tanımlı değil.")

if "val_df_filtered" in globals():
    val_eval_df = val_df_filtered.copy().reset_index(drop=True)
    val_source_name = "val_df_filtered"
elif "val_df" in globals():
    val_eval_df = val_df.copy().reset_index(drop=True)
    val_source_name = "val_df"
else:
    raise RuntimeError("Validation dataframe bulunamadı. Önce split/eğitim hücrelerini çalıştırın.")

ckpt_candidates = []
if "FILTERED_PATHS" in globals():
    ckpt_candidates.append(Path(FILTERED_PATHS["best_model_path"]))
if "filtered_results" in globals() and isinstance(filtered_results, dict):
    run_dir = filtered_results.get("run_dir", None)
    if run_dir is not None:
        ckpt_candidates.append(Path(run_dir) / "best_model.pth")
if "PATHS" in globals():
    ckpt_candidates.append(Path(PATHS["best_model_path"]))
ckpt_candidates.extend([
    Path.cwd() / "deeplabv3+" / "best_model.pth",
    Path.cwd() / "best_model.pth",
])

best_ckpt_path = next((p for p in ckpt_candidates if Path(p).exists()), None)
if best_ckpt_path is None:
    raise FileNotFoundError("En iyi model ağırlığı (`best_model.pth`) bulunamadı.")

ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
viz_cfg = dict(filtered_cfg) if "filtered_cfg" in globals() else dict(CFG)
if isinstance(ckpt, dict) and "cfg" in ckpt and isinstance(ckpt["cfg"], dict):
    viz_cfg.update(ckpt["cfg"])

model = build_model(viz_cfg).to(DEVICE)
state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
model.load_state_dict(state_dict)
model.eval()

best_threshold = float(ckpt.get("threshold", 0.5)) if isinstance(ckpt, dict) else 0.5
min_area = int(viz_cfg.get("min_component_area", 0))
use_tta = bool(viz_cfg.get("tta_inference", False))

auth_df = val_eval_df[val_eval_df["label"] == "authentic"].copy().reset_index(drop=True)
forged_df = val_eval_df[val_eval_df["label"] == "forged"].copy().reset_index(drop=True)
if "gt_positive_pixels" in forged_df.columns:
    forged_df = forged_df[forged_df["gt_positive_pixels"] > 0].copy().reset_index(drop=True)

rng = np.random.default_rng(viz_cfg.get("seed", 42))
auth_pick_n = min(3, len(auth_df))
forged_pick_n = min(3, len(forged_df))

selected_frames = []
if auth_pick_n > 0:
    auth_idx = rng.choice(len(auth_df), size=auth_pick_n, replace=False)
    selected_frames.append(auth_df.iloc[auth_idx].copy())
if forged_pick_n > 0:
    forged_idx = rng.choice(len(forged_df), size=forged_pick_n, replace=False)
    selected_frames.append(forged_df.iloc[forged_idx].copy())

if not selected_frames:
    raise RuntimeError("Gösterilecek validation örneği bulunamadı.")

selected_examples_df = pd.concat(selected_frames, axis=0).reset_index(drop=True)
display(selected_examples_df[[col for col in ["case_id", "label", "source", "has_mask", "gt_positive_pixels", "image_path"] if col in selected_examples_df.columns]])

save_root = FILTERED_PATHS["visualizations_dir"] if "FILTERED_PATHS" in globals() else PATHS["visualizations_dir"]
save_dir = Path(save_root) / "best_model_validation_examples"
save_dir.mkdir(parents=True, exist_ok=True)

print(f"Checkpoint: {best_ckpt_path}")
print(f"Validation kaynağı: {val_source_name}")
print(f"Threshold: {best_threshold:.2f} | TTA: {use_tta}")
print(f"Gösterilen örnek sayısı -> authentic: {auth_pick_n}, forged: {forged_pick_n}")
print(f"Görseller kaydediliyor -> {save_dir}")

@torch.no_grad()
def visualize_validation_predictions(model, df, cfg, device, threshold, save_dir, use_tta=False):
    for idx, row in df.reset_index(drop=True).iterrows():
        image = read_image_rgb(row["image_path"])

        mask_paths = row.get("mask_paths", [])
        if isinstance(mask_paths, (list, tuple)) and len(mask_paths) > 0 and int(row.get("has_mask", 0)) == 1:
            gt_mask = union_masks(mask_paths, image.shape[:2])
        else:
            gt_mask = np.zeros(image.shape[:2], dtype=np.uint8)

        one_item_df = pd.DataFrame([row.to_dict()])
        one_item_ds = ForgeryDataset(one_item_df, cfg, is_train=False, with_masks=True)
        sample = one_item_ds[0]
        image_tensor = sample["image"].unsqueeze(0).to(device)

        prob = predict_with_tta(model, image_tensor, use_tta=use_tta)[0, 0].detach().cpu().numpy()
        prob = cv2.resize(prob, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_LINEAR)
        pred_mask = postprocess_mask((prob >= threshold).astype(np.uint8), min_area=min_area)

        fig, axes = plt.subplots(1, 5, figsize=(18, 4))
        axes[0].imshow(image)
        axes[0].set_title(f"image | {row['label']} | case_id={row['case_id']}")
        axes[1].imshow(gt_mask, cmap="gray")
        axes[1].set_title("ground truth")
        axes[2].imshow(prob, cmap="magma")
        axes[2].set_title("probability")
        axes[3].imshow(pred_mask, cmap="gray")
        axes[3].set_title(f"prediction @ {threshold:.2f}")
        axes[4].imshow(make_overlay(image, pred_mask))
        axes[4].set_title("overlay")

        for ax in axes:
            ax.axis("off")

        plt.tight_layout()
        fig.savefig(save_dir / f"{idx + 1:02d}_{row['label']}_case_{int(row['case_id'])}.png", dpi=150)
        plt.show()
        plt.close(fig)


visualize_validation_predictions(
    model=model,
    df=selected_examples_df,
    cfg=viz_cfg,
    device=DEVICE,
    threshold=best_threshold,
    save_dir=save_dir,
    use_tta=use_tta,
)

In [ ]:
# Eğitim sonrası — sınıf bazlı aksiyon metrikleri (validation)
# Bu hücre tüm validation seti üzerinde en iyi checkpoint ile inference yapar
# ve class-level (authentic / forged) özetler üretir.

if "DEVICE" not in globals():
    raise RuntimeError("Önce eğitim hücrelerini çalıştırın; `DEVICE` henüz tanımlı değil.")

if "val_df_filtered" in globals():
    eval_df = val_df_filtered.copy().reset_index(drop=True)
    eval_df_name = "val_df_filtered"
elif "val_df" in globals():
    eval_df = val_df.copy().reset_index(drop=True)
    eval_df_name = "val_df"
else:
    raise RuntimeError("Validation dataframe bulunamadı. Önce split/eğitim hücrelerini çalıştırın.")

ckpt_candidates = []
if "FILTERED_PATHS" in globals():
    ckpt_candidates.append(Path(FILTERED_PATHS["best_model_path"]))
if "filtered_results" in globals() and isinstance(filtered_results, dict):
    if filtered_results.get("run_dir"):
        ckpt_candidates.append(Path(filtered_results["run_dir"]) / "best_model.pth")
if "PATHS" in globals():
    ckpt_candidates.append(Path(PATHS["best_model_path"]))
ckpt_candidates.extend([
    Path.cwd() / "deeplabv3+" / "best_model.pth",
    Path.cwd() / "best_model.pth",
])

best_ckpt_path = next((p for p in ckpt_candidates if Path(p).exists()), None)
if best_ckpt_path is None:
    raise FileNotFoundError("En iyi model ağırlığı (`best_model.pth`) bulunamadı.")

ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
diag_cfg = dict(filtered_cfg) if "filtered_cfg" in globals() else dict(CFG)
if isinstance(ckpt, dict) and "cfg" in ckpt and isinstance(ckpt["cfg"], dict):
    diag_cfg.update(ckpt["cfg"])

model = build_model(diag_cfg).to(DEVICE)
state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
model.load_state_dict(state_dict)
model.eval()

best_threshold = float(ckpt.get("threshold", 0.5)) if isinstance(ckpt, dict) else 0.5
min_area = int(diag_cfg.get("min_component_area", 0))
use_tta = bool(diag_cfg.get("tta_inference", False))


@torch.no_grad()
def safe_predict_prob_map(model, image_tensor, use_tta=False):
    if "predict_with_tta" in globals():
        pred = predict_with_tta(model, image_tensor, use_tta=use_tta)
    else:
        pred = model(image_tensor)
        if isinstance(pred, (list, tuple)):
            pred = pred[0]
        pred = torch.sigmoid(pred)
    return pred[0, 0].detach().cpu().numpy()


@torch.no_grad()
def collect_case_level_metrics(model, df, cfg, device, threshold, min_area=0, use_tta=False):
    rows = []
    for row in tqdm(df.itertuples(index=False), total=len(df), desc="validation diagnostics"):
        image = read_image_rgb(row.image_path)
        mask_paths = list(getattr(row, "mask_paths", [])) if getattr(row, "mask_paths", None) is not None else []

        if int(getattr(row, "has_mask", 0)) == 1 and len(mask_paths) > 0:
            gt_mask = union_masks(mask_paths, image.shape[:2])
        else:
            gt_mask = np.zeros(image.shape[:2], dtype=np.uint8)

        one_item_df = pd.DataFrame([row._asdict()])
        one_item_ds = ForgeryDataset(one_item_df, cfg, is_train=False, with_masks=True)
        sample = one_item_ds[0]
        image_tensor = sample["image"].unsqueeze(0).to(device)

        prob = safe_predict_prob_map(model, image_tensor, use_tta=use_tta)
        prob = cv2.resize(prob, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_LINEAR)
        pred_mask = postprocess_mask((prob >= threshold).astype(np.uint8), min_area=min_area)

        gt_pos = int(gt_mask.sum())
        pred_pos = int(pred_mask.sum())
        pixel_stats = pixel_metrics_from_arrays(gt_mask, pred_mask)

        rows.append({
            "case_id": int(row.case_id),
            "label": row.label,
            "source": getattr(row, "source", "unknown"),
            "has_mask": int(getattr(row, "has_mask", 0)),
            "gt_positive_pixels": gt_pos,
            "pred_positive_pixels": pred_pos,
            "gt_is_empty": int(gt_pos == 0),
            "pred_is_empty": int(pred_pos == 0),
            "image_level_tp": int(gt_pos > 0 and pred_pos > 0),
            "image_level_tn": int(gt_pos == 0 and pred_pos == 0),
            "image_level_fp": int(gt_pos == 0 and pred_pos > 0),
            "image_level_fn": int(gt_pos > 0 and pred_pos == 0),
            "pred_to_gt_area_ratio": float(pred_pos / max(gt_pos, 1)),
            "prob_mean": float(prob.mean()),
            "prob_max": float(prob.max()),
            **pixel_stats,
        })

    return pd.DataFrame(rows)


case_metrics_df = collect_case_level_metrics(
    model=model,
    df=eval_df,
    cfg=diag_cfg,
    device=DEVICE,
    threshold=best_threshold,
    min_area=min_area,
    use_tta=use_tta,
)

auth_df = case_metrics_df[case_metrics_df["label"] == "authentic"].copy().reset_index(drop=True)
forged_df = case_metrics_df[case_metrics_df["label"] == "forged"].copy().reset_index(drop=True)
forged_nonempty_df = forged_df[forged_df["gt_positive_pixels"] > 0].copy().reset_index(drop=True)

class_summary_df = pd.DataFrame([
    {
        "class": "authentic",
        "n_images": int(len(auth_df)),
        "empty_prediction_rate": float((auth_df["pred_is_empty"] == 1).mean()) if len(auth_df) else np.nan,
        "false_alarm_rate": float((auth_df["image_level_fp"] == 1).mean()) if len(auth_df) else np.nan,
        "avg_pred_positive_pixels": float(auth_df["pred_positive_pixels"].mean()) if len(auth_df) else np.nan,
        "avg_prob_max": float(auth_df["prob_max"].mean()) if len(auth_df) else np.nan,
    },
    {
        "class": "forged_nonempty_gt",
        "n_images": int(len(forged_nonempty_df)),
        "image_detect_rate": float((forged_nonempty_df["image_level_tp"] == 1).mean()) if len(forged_nonempty_df) else np.nan,
        "image_miss_rate": float((forged_nonempty_df["image_level_fn"] == 1).mean()) if len(forged_nonempty_df) else np.nan,
        "mean_dice": float(forged_nonempty_df["dice"].mean()) if len(forged_nonempty_df) else np.nan,
        "median_dice": float(forged_nonempty_df["dice"].median()) if len(forged_nonempty_df) else np.nan,
        "mean_iou": float(forged_nonempty_df["iou"].mean()) if len(forged_nonempty_df) else np.nan,
        "mean_precision": float(forged_nonempty_df["precision"].mean()) if len(forged_nonempty_df) else np.nan,
        "mean_recall": float(forged_nonempty_df["recall"].mean()) if len(forged_nonempty_df) else np.nan,
        "avg_pred_to_gt_area_ratio": float(forged_nonempty_df["pred_to_gt_area_ratio"].mean()) if len(forged_nonempty_df) else np.nan,
    },
])

display(class_summary_df)

hard_auth_cases = auth_df.sort_values(["pred_positive_pixels", "prob_max"], ascending=[False, False]).head(10)
hard_forged_cases = forged_nonempty_df.sort_values(["dice", "recall", "precision"], ascending=[True, True, True]).head(10)

print(f"Validation dataframe: {eval_df_name} | threshold={best_threshold:.2f} | TTA={use_tta}")
print(f"Authentic görüntülerde false alarm rate: {((auth_df['image_level_fp'] == 1).mean() if len(auth_df) else 0.0):.4f}")
print(f"Forged görüntülerde image-level detect rate: {((forged_nonempty_df['image_level_tp'] == 1).mean() if len(forged_nonempty_df) else 0.0):.4f}")

print("\nEn problemli authentic örnekler (yanlış pozitif riski yüksek):")
display(hard_auth_cases[[col for col in ["case_id", "source", "pred_positive_pixels", "prob_max", "prob_mean"] if col in hard_auth_cases.columns]])

print("En problemli forged örnekler (kaçırılan / kötü lokalize edilen):")
display(hard_forged_cases[[col for col in ["case_id", "source", "gt_positive_pixels", "pred_positive_pixels", "dice", "iou", "precision", "recall"] if col in hard_forged_cases.columns]])

action_rows = []
auth_false_alarm_rate = float((auth_df["image_level_fp"] == 1).mean()) if len(auth_df) else 0.0
forged_detect_rate = float((forged_nonempty_df["image_level_tp"] == 1).mean()) if len(forged_nonempty_df) else 0.0
forged_mean_dice = float(forged_nonempty_df["dice"].mean()) if len(forged_nonempty_df) else 0.0
forged_mean_precision = float(forged_nonempty_df["precision"].mean()) if len(forged_nonempty_df) else 0.0
forged_mean_recall = float(forged_nonempty_df["recall"].mean()) if len(forged_nonempty_df) else 0.0

if auth_false_alarm_rate > 0.10:
    action_rows.append({
        "priority": "high",
        "observation": "Authentic görüntülerde false positive yüksek.",
        "recommended_action": "Threshold / post-processing sıkılaştır, hard negative mining yap, authentic örnekler için daha hedefli augmentations dene.",
    })
if forged_mean_precision < 0.20:
    action_rows.append({
        "priority": "high",
        "observation": "Forged sınıfında precision çok düşük; model fazla alanı boyuyor.",
        "recommended_action": "`min_component_area` artır, Focal/Tversky loss dene, küçük gürültü bileşenlerini silen morfolojik post-processing ekle.",
    })
if forged_mean_recall < 0.50:
    action_rows.append({
        "priority": "medium",
        "observation": "Forged alanların önemli kısmı kaçıyor.",
        "recommended_action": "Daha fazla pozitif örnek, crop-based training, pozitif ağırlık / sampler ayarı ve uygun augmentation dene.",
    })
if forged_mean_dice < 0.20:
    action_rows.append({
        "priority": "high",
        "observation": "Lokalizasyon kalitesi düşük (`mean_dice < 0.20`).",
        "recommended_action": "Veri/mask kalite kontrolü yap, hatalı veya aşırı küçük GT örneklerini incele, farklı loss ve encoder kombinasyonları dene.",
    })
if forged_detect_rate >= 0.70 and forged_mean_precision < 0.20:
    action_rows.append({
        "priority": "medium",
        "observation": "Model sahte görüntüyü genelde buluyor ama maskeyi geniş boyuyor.",
        "recommended_action": "Daha yüksek threshold, boundary-aware loss veya refined post-processing ile alanı daralt.",
    })
if not action_rows:
    action_rows.append({
        "priority": "info",
        "observation": "Belirgin alarm seviyesi görülmedi.",
        "recommended_action": "Sonuçları hata örnekleri üzerinden görsel olarak inceleyerek sonraki deney planını belirle.",
    })

action_plan_df = pd.DataFrame(action_rows)
print("\nAksiyon özeti:")
display(action_plan_df)

analysis_root = FILTERED_PATHS["run_dir"] if "FILTERED_PATHS" in globals() else PATHS["run_dir"]
analysis_dir = Path(analysis_root) / "post_train_diagnostics"
analysis_dir.mkdir(parents=True, exist_ok=True)
case_metrics_df.to_csv(analysis_dir / "validation_case_metrics.csv", index=False)
class_summary_df.to_csv(analysis_dir / "validation_class_summary.csv", index=False)
action_plan_df.to_csv(analysis_dir / "validation_action_plan.csv", index=False)
print(f"Tanı dosyaları kaydedildi -> {analysis_dir}")

In [ ]:
# Veri seti yapısı + kalite denetimi (bütüncül analiz)
# Bu hücre, veri setinin dağılımını, maske kalitesini, çözünürlük yapısını
# ve aksiyon gerektiren şüpheli kayıtları özetler.

if "DATA_ROOT" not in globals():
    raise RuntimeError("Önce veri hazırlama hücrelerini çalıştırın; `DATA_ROOT` henüz tanımlı değil.")

if "samples_df" not in globals():
    samples_df = build_samples_dataframe(DATA_ROOT)

dataset_audit_df = samples_df.copy().reset_index(drop=True)
if "gt_positive_pixels" not in dataset_audit_df.columns or "has_nonempty_gt" not in dataset_audit_df.columns:
    dataset_audit_df = annotate_ground_truth_columns(dataset_audit_df)

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"Toplam eğitim/supplemental örnek sayısı: {len(dataset_audit_df)}")

source_label_summary_df = (
    dataset_audit_df
    .groupby(["source", "label"], dropna=False)
    .agg(
        n_images=("image_path", "count"),
        unique_case_ids=("case_id", "nunique"),
        with_mask_files=("has_mask", "sum"),
        nonempty_gt=("has_nonempty_gt", "sum"),
    )
    .reset_index()
    .sort_values(["source", "label"])
    .reset_index(drop=True)
)

label_overall_summary_df = pd.DataFrame([
    {
        "metric": "total_images",
        "value": int(len(dataset_audit_df)),
    },
    {
        "metric": "total_unique_case_ids",
        "value": int(dataset_audit_df["case_id"].nunique()),
    },
    {
        "metric": "authentic_images",
        "value": int((dataset_audit_df["label"] == "authentic").sum()),
    },
    {
        "metric": "forged_images",
        "value": int((dataset_audit_df["label"] == "forged").sum()),
    },
    {
        "metric": "forged_with_mask_file",
        "value": int(((dataset_audit_df["label"] == "forged") & (dataset_audit_df["has_mask"] == 1)).sum()),
    },
    {
        "metric": "forged_with_nonempty_gt",
        "value": int(((dataset_audit_df["label"] == "forged") & (dataset_audit_df["gt_positive_pixels"] > 0)).sum()),
    },
    {
        "metric": "forged_with_empty_or_missing_gt",
        "value": int(((dataset_audit_df["label"] == "forged") & (dataset_audit_df["gt_positive_pixels"] == 0)).sum()),
    },
])

print("\n1) Kaynak ve etiket bazlı dağılım")
display(source_label_summary_df)
print("2) Genel özet")
display(label_overall_summary_df)


# Görsel metadata taraması (çözünürlük / kanal / okunabilirlik)
image_meta_rows = []
for row in tqdm(dataset_audit_df[["case_id", "label", "source", "image_path"]].drop_duplicates().itertuples(index=False),
                total=dataset_audit_df[["image_path"]].drop_duplicates().shape[0],
                desc="image metadata scan"):
    img = cv2.imread(str(row.image_path), cv2.IMREAD_UNCHANGED)
    if img is None:
        image_meta_rows.append({
            "case_id": int(row.case_id),
            "label": row.label,
            "source": row.source,
            "image_path": row.image_path,
            "read_ok": 0,
            "height": np.nan,
            "width": np.nan,
            "channels": np.nan,
            "dtype": None,
        })
        continue

    h, w = img.shape[:2]
    channels = 1 if img.ndim == 2 else int(img.shape[2])
    image_meta_rows.append({
        "case_id": int(row.case_id),
        "label": row.label,
        "source": row.source,
        "image_path": row.image_path,
        "read_ok": 1,
        "height": int(h),
        "width": int(w),
        "channels": channels,
        "dtype": str(img.dtype),
    })

image_meta_df = pd.DataFrame(image_meta_rows)
dataset_audit_df = dataset_audit_df.merge(
    image_meta_df[["image_path", "read_ok", "height", "width", "channels", "dtype"]],
    on="image_path",
    how="left",
)
dataset_audit_df["image_area"] = dataset_audit_df["height"].fillna(0) * dataset_audit_df["width"].fillna(0)
dataset_audit_df["gt_area_fraction"] = dataset_audit_df["gt_positive_pixels"] / np.maximum(dataset_audit_df["image_area"], 1)

resolution_summary_df = pd.DataFrame([
    {
        "n_images_scanned": int(len(image_meta_df)),
        "unreadable_images": int((image_meta_df["read_ok"] == 0).sum()),
        "height_min": int(image_meta_df["height"].min()) if image_meta_df["height"].notna().any() else None,
        "height_median": float(image_meta_df["height"].median()) if image_meta_df["height"].notna().any() else None,
        "height_max": int(image_meta_df["height"].max()) if image_meta_df["height"].notna().any() else None,
        "width_min": int(image_meta_df["width"].min()) if image_meta_df["width"].notna().any() else None,
        "width_median": float(image_meta_df["width"].median()) if image_meta_df["width"].notna().any() else None,
        "width_max": int(image_meta_df["width"].max()) if image_meta_df["width"].notna().any() else None,
        "n_unique_resolutions": int(image_meta_df[["height", "width"]].dropna().drop_duplicates().shape[0]),
    }
])

top_resolutions_df = (
    image_meta_df[image_meta_df["read_ok"] == 1]
    .groupby(["height", "width", "channels"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

print("3) Çözünürlük / okuma özeti")
display(resolution_summary_df)
print("En sık görülen çözünürlükler")
display(top_resolutions_df)


# Maske kalite analizi
forged_all_df = dataset_audit_df[dataset_audit_df["label"] == "forged"].copy().reset_index(drop=True)
forged_nonempty_df = forged_all_df[forged_all_df["gt_positive_pixels"] > 0].copy().reset_index(drop=True)
tiny_threshold = 0.001  # görüntünün %0.1'inden küçük maskeler

mask_quality_summary_df = pd.DataFrame([
    {
        "forged_total": int(len(forged_all_df)),
        "forged_without_mask_file": int(((forged_all_df["has_mask"] == 0)).sum()),
        "forged_with_empty_gt": int((forged_all_df["gt_positive_pixels"] == 0).sum()),
        "forged_with_nonempty_gt": int((forged_all_df["gt_positive_pixels"] > 0).sum()),
        "tiny_masks_lt_0.1pct": int((forged_nonempty_df["gt_area_fraction"] < tiny_threshold).sum()) if len(forged_nonempty_df) else 0,
        "median_gt_positive_pixels": float(forged_nonempty_df["gt_positive_pixels"].median()) if len(forged_nonempty_df) else np.nan,
        "median_gt_area_fraction": float(forged_nonempty_df["gt_area_fraction"].median()) if len(forged_nonempty_df) else np.nan,
        "max_gt_area_fraction": float(forged_nonempty_df["gt_area_fraction"].max()) if len(forged_nonempty_df) else np.nan,
    }
])

print("4) Forged mask kalite özeti")
display(mask_quality_summary_df)


# Case-id ve tekrar yapısı
case_cardinality_df = (
    dataset_audit_df.groupby("case_id")
    .agg(n_images=("image_path", "count"), labels=("label", lambda x: ", ".join(sorted(set(map(str, x))))))
    .reset_index()
)

case_multiplicity_summary_df = pd.DataFrame([
    {
        "n_case_ids": int(len(case_cardinality_df)),
        "single_image_cases": int((case_cardinality_df["n_images"] == 1).sum()),
        "multi_image_cases": int((case_cardinality_df["n_images"] > 1).sum()),
        "max_images_per_case": int(case_cardinality_df["n_images"].max()),
        "mean_images_per_case": float(case_cardinality_df["n_images"].mean()),
    }
])

print("5) Case yapısı")
display(case_multiplicity_summary_df)


# Şüpheli / aksiyon alınabilecek kayıtlar
missing_mask_df = forged_all_df[forged_all_df["has_mask"] == 0].copy().reset_index(drop=True)
empty_gt_df = forged_all_df[(forged_all_df["has_mask"] == 1) & (forged_all_df["gt_positive_pixels"] == 0)].copy().reset_index(drop=True)
tiny_gt_df = forged_nonempty_df[forged_nonempty_df["gt_area_fraction"] < tiny_threshold].copy().reset_index(drop=True)
unreadable_df = dataset_audit_df[dataset_audit_df["read_ok"] == 0].copy().reset_index(drop=True)

print("6) Şüpheli forged kayıtları — maskesi eksik")
display(missing_mask_df[[col for col in ["case_id", "source", "label", "image_path"] if col in missing_mask_df.columns]].head(20))
print("7) Şüpheli forged kayıtları — GT boş")
display(empty_gt_df[[col for col in ["case_id", "source", "gt_positive_pixels", "image_path"] if col in empty_gt_df.columns]].head(20))
print("8) Çok küçük GT maskeleri")
display(tiny_gt_df[[col for col in ["case_id", "source", "gt_positive_pixels", "gt_area_fraction", "image_path"] if col in tiny_gt_df.columns]].head(20))
if len(unreadable_df) > 0:
    print("9) Okunamayan görüntüler")
    display(unreadable_df[[col for col in ["case_id", "source", "label", "image_path"] if col in unreadable_df.columns]].head(20))


# Veri seti için aksiyon önerileri
recommendations = []
auth_count = int((dataset_audit_df["label"] == "authentic").sum())
forged_count = int((dataset_audit_df["label"] == "forged").sum())
nonempty_forged_count = int((dataset_audit_df["gt_positive_pixels"] > 0).sum())
imbalance_ratio = nonempty_forged_count / max(auth_count, 1)

if len(missing_mask_df) > 0:
    recommendations.append({
        "priority": "high",
        "issue": "Forged etiketli ama maskesi eksik örnekler var.",
        "recommended_action": "Bu kayıtları ya temizle ya da classification-only olarak ayrı ele al; segmentation eğitimine direkt dahil etme.",
    })
if len(empty_gt_df) > 0:
    recommendations.append({
        "priority": "high",
        "issue": "Forged etiketli fakat GT maskesi boş örnekler var.",
        "recommended_action": "Maske üretim sürecini doğrula; bunlar label noise oluşturup Dice/IoU'yu ciddi düşürür.",
    })
if len(tiny_gt_df) > 0:
    recommendations.append({
        "priority": "medium",
        "issue": "Çok küçük sahte alanlar mevcut.",
        "recommended_action": "Patch/crop tabanlı eğitim, focal-type loss ve daha yüksek çözünürlük ile dene.",
    })
if imbalance_ratio < 0.20:
    recommendations.append({
        "priority": "medium",
        "issue": "Authentic vs non-empty forged dengesizliği yüksek.",
        "recommended_action": "Sampler, class weighting ve forged-heavy augmentation ile dengele.",
    })
if int((image_meta_df[["height", "width"]].dropna().drop_duplicates().shape[0])) > 10:
    recommendations.append({
        "priority": "medium",
        "issue": "Çözünürlük çeşitliliği yüksek.",
        "recommended_action": "Aspect-ratio koruyan resize/pad stratejisini ve multi-scale training yaklaşımını gözden geçir.",
    })
if len(unreadable_df) > 0:
    recommendations.append({
        "priority": "high",
        "issue": "Okunamayan dosyalar var.",
        "recommended_action": "Bu dosyaları veri setinden çıkar veya yeniden oluştur.",
    })
if not recommendations:
    recommendations.append({
        "priority": "info",
        "issue": "Belirgin veri kalitesi alarmı görülmedi.",
        "recommended_action": "Hata analizi ve model/loss tuning aşamasına geçilebilir.",
    })

dataset_action_plan_df = pd.DataFrame(recommendations)
print("10) Veri seti için aksiyon planı")
display(dataset_action_plan_df)

analysis_root = FILTERED_PATHS["run_dir"] if "FILTERED_PATHS" in globals() else PATHS["run_dir"]
audit_dir = Path(analysis_root) / "dataset_diagnostics"
audit_dir.mkdir(parents=True, exist_ok=True)
source_label_summary_df.to_csv(audit_dir / "source_label_summary.csv", index=False)
label_overall_summary_df.to_csv(audit_dir / "label_overall_summary.csv", index=False)
resolution_summary_df.to_csv(audit_dir / "resolution_summary.csv", index=False)
top_resolutions_df.to_csv(audit_dir / "top_resolutions.csv", index=False)
mask_quality_summary_df.to_csv(audit_dir / "mask_quality_summary.csv", index=False)
dataset_action_plan_df.to_csv(audit_dir / "dataset_action_plan.csv", index=False)
missing_mask_df.to_csv(audit_dir / "suspicious_missing_mask.csv", index=False)
empty_gt_df.to_csv(audit_dir / "suspicious_empty_gt.csv", index=False)
tiny_gt_df.to_csv(audit_dir / "suspicious_tiny_gt.csv", index=False)
print(f"Dataset audit çıktıları kaydedildi -> {audit_dir}")